# RE-Matching — FewRel, m = 5

Notebook này chỉ chạy **m = 5**, gồm **5 lượt độc lập** trên các split `ori`, `40`, `41`, `42`, `43`.
Không chạy m = 10. Mỗi lượt khởi tạo lại model, train rồi test; tổng hợp mean và sample std
riêng cho mỗi m. Seed train được giữ cố định, còn split quan hệ thay đổi.

**Trước khi Run All:**
1. Commit/push bản code đã refactor lên fork GitHub; notebook kiểm tra repo có CLI mới.
2. Bật **Internet** và **GPU accelerator** trong Kaggle. Dataset FewRel được lấy từ repo, không cần attach thêm.
3. Cấu hình đã đặt sẵn theo script m = 10 và defaults trong `model/train.py`: batch size 128, epochs 5, seed 42, lr 2e-6. Không cần sửa để chạy cấu hình này.
4. Bộ requirements hiện tại cần **Python >= 3.12**. Notebook giữ bản PyTorch CUDA có sẵn.

Kết quả trong `/kaggle/working/fewrel_m5`: log, checkpoint, manifest, CSV/JSON.
Lưu một Kaggle Version **có outputs** sau phiên chạy. Để tiếp tục ở phiên mới, attach outputs
cũ làm input rồi đặt `PREVIOUS_OUTPUT_DIR`. Notebook bỏ qua lượt **đã hoàn tất và khớp cấu hình**;
lượt đang dở sẽ train lại từ đầu, không resume optimizer giữa epoch.

Code hiện tại dùng split `eval` để tách quan hệ nhưng chưa thực hiện chọn siêu tham số trên
validation. Notebook giữ hành vi train/test của repo, không tự thêm quy trình tuning.


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import os
import re
import shutil
import statistics
import subprocess
import sys
import time
from datetime import datetime, timezone

REPO_URL = "https://github.com/ncmanh-uet/RE-Matching.git"
REPO_REF = "d8bc273fc1ddc45fc0cf6a6aa6c4c06841914c60"  # Commit chứa code refactor và dataset FewRel.
REPO_DIR = Path("/kaggle/working/RE-Matching")
DATASET_FILE = ""  # Rỗng = ưu tiên data/fewrel trong repo, sau đó tìm /kaggle/input.
OUTPUT_DIR = Path("/kaggle/working/fewrel_m5")
PREVIOUS_OUTPUT_DIR = ""  # Thư mục fewrel_m5 đã attach từ outputs phiên trước.

M_VALUES = [5]
SPLITS = ["ori", "40", "41", "42", "43"]
# Cấu hình m=10 theo run_fewrel.sh + defaults model/train.py.
# Không có log Kaggle m=10 trong repo để xác minh các override riêng trước đây.
TRAIN_SEED = 42
EPOCHS = 5
BATCH_SIZE = 128  # Đã chốt theo defaults của script m=10 trong repo.
EVAL_BATCH_SIZE = 32
VISIBLE_DEVICES = "0"  # "0,1" nếu muốn dùng DataParallel trên 2 GPU.
MAX_LENGTH = 512
LR = 2e-6
ALPHA = 0.33
GAMMA = 0.06
NUM_NEGSAMPLE = 7
WARM_UP = 0.1
PRETRAINED_MODEL = "bert-base-uncased"
DESCRIPTION_MODEL = "sentence-transformers/stsb-bert-base"
RELATION_DESCRIPTION = "relation_description_addsynonym_processed.json"
RUN_SMOKE_TEST = True  # Hai bước train + đánh giá toàn test split của m đầu tiên.


## 1. Lấy code và cài dependency
Không pull tự động nếu thư mục repo đã tồn tại. Repo, commit và các phiên bản thực tế được
ghi vào manifest. Cài thư viện qua subprocess, train cũng chạy ở subprocess riêng để tránh
vấn đề kernel đã import phiên bản thư viện cũ.


In [ ]:
assert sys.version_info >= (3, 12), f"Requirements này cần Python >=3.12, hiện là {sys.version}"
assert M_VALUES and set(M_VALUES) <= {5, 15}, "Notebook này chỉ chạy m=5 và m=15"
assert SPLITS and len(SPLITS) == len(set(SPLITS))
assert set(SPLITS) <= {"ori", "40", "41", "42", "43"}
assert BATCH_SIZE > 0 and EPOCHS > 0
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
if REPO_REF:
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", REPO_REF], cwd=REPO_DIR, check=True)
train_source = (REPO_DIR / "model/train.py").read_text()
assert "--ckpt_dir" in train_source and "--max_steps" in train_source, (
    "Repo chưa có code refactor. Push code mới và đặt REPO_REF đúng commit."
)
GPU_CHECK = "import torch; assert torch.cuda.is_available(), 'Kaggle GPU unavailable'; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name())"
subprocess.run([sys.executable, "-c", GPU_CHECK], check=True)
# Prevent pip from silently replacing the CUDA-enabled torch build.
torch_version = subprocess.check_output(
    [sys.executable, "-c", "import torch; print(torch.__version__)"], text=True).strip()
constraint = REPO_DIR / "kaggle-torch-constraint.txt"
constraint.write_text(f"torch=={torch_version}\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt"),
                "-c", str(constraint)], check=True)
subprocess.run([sys.executable, "-c", GPU_CHECK], check=True)
ENVIRONMENT = json.loads(subprocess.check_output([sys.executable, "-c", """
import json, sys, torch, transformers, sentence_transformers, numpy, sklearn
from importlib.metadata import version
print(json.dumps({'python': sys.version, 'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(), 'torch_build': torch.__version__,
    'versions': {p: version(p) for p in ['torch', 'transformers', 'sentence-transformers', 'numpy', 'scikit-learn']}}))
"""], text=True))
print(json.dumps(ENVIRONMENT, indent=2))


## 2. Chuẩn bị dữ liệu và kiểm tra split
Dataset cần là bản processed của repo (`token`, `relation`, `h.pos`, `t.pos`), không phải
FewRel raw dạng dictionary. Không trộn sample giữa train/test: chia theo relation ID.


In [ ]:
if DATASET_FILE:
    source_dataset = Path(DATASET_FILE)
elif (REPO_DIR / "data/fewrel/fewrel_dataset.json").is_file():
    source_dataset = REPO_DIR / "data/fewrel/fewrel_dataset.json"
else:
    matches = sorted(Path("/kaggle/input").rglob("fewrel_dataset.json"))
    if len(matches) != 1:
        raise ValueError(f"Tìm thấy {len(matches)} dataset: {matches}. Hãy đặt DATASET_FILE.")
    source_dataset = matches[0]
assert source_dataset.is_file(), source_dataset
DATA_ROOT = Path("/kaggle/working/rematching-data")
(DATA_ROOT / "fewrel").mkdir(parents=True, exist_ok=True)
dataset_target = DATA_ROOT / "fewrel/fewrel_dataset.json"
if source_dataset.resolve() != dataset_target.resolve():
    shutil.copy2(source_dataset, dataset_target)
description_dir = REPO_DIR / "data/fewrel/relation_description"
shutil.copytree(description_dir, DATA_ROOT / "fewrel/relation_description", dirs_exist_ok=True)
REL2ID_ROOT = REPO_DIR / "data/rel2id"
raw = json.loads(dataset_target.read_text())
assert isinstance(raw, list) and raw, "Cần JSON list của FewRel đã xử lý."
relations = set()
for idx, item in enumerate(raw):
    assert isinstance(item.get('token'), list) and item['token'], f"Invalid tokens: sample {idx}"
    spans = [item['h']['pos'], item['t']['pos']]
    for start, end in spans:
        assert 0 <= start < end <= len(item['token']), f"Invalid span: sample {idx}"
    assert spans[0][1] <= spans[1][0] or spans[1][1] <= spans[0][0], f"Overlapping spans: sample {idx}"
    relations.add(item['relation'])
assert len(relations) == 80, f"FewRel mong đợi 80 quan hệ, nhận {len(relations)}"
description_relations = {d['relation'] for d in json.loads(
    (DATA_ROOT / 'fewrel/relation_description' / RELATION_DESCRIPTION).read_text())}
SPLIT_INFO = []
for m in M_VALUES:
    for split in SPLITS:
        path = REL2ID_ROOT / f"fewrel_rel2id/fewrel_rel2id_{m}_{split}.json"
        partition = json.loads(path.read_text())
        train, val, test = [set(partition[k]) for k in ['train', 'eval', 'test']]
        assert (len(train), len(val), len(test)) == (75 - m, 5, m), path
        assert not (train & val or train & test or val & test), path
        assert train | val | test == relations, path
        assert train | test <= description_relations, path
        SPLIT_INFO.append({'m': m, 'split': split, 'train_relations': len(train),
                           'validation_relations': len(val), 'test_relations': len(test)})
print(f"Dataset: {len(raw):,} samples, {len(relations)} relations")
del raw
print(json.dumps(SPLIT_INFO, indent=2))


## 3. Lưu cấu hình và khôi phục các lượt đã hoàn tất
Không tự giảm batch size khi gặp OOM vì batch size ảnh hưởng negative sampling. Nếu đổi
cấu hình hoặc code, hãy dùng `OUTPUT_DIR` mới. Không so sánh hoặc gộp lẫn kết quả khác cấu hình.


In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(4 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if PREVIOUS_OUTPUT_DIR:
    previous = Path(PREVIOUS_OUTPUT_DIR)
    assert previous.is_dir() and (previous / 'manifest.json').is_file(), previous
    if previous.resolve() != OUTPUT_DIR.resolve():
        assert not any(OUTPUT_DIR.iterdir()), 'Khôi phục vào OUTPUT_DIR trống để tránh ghi đè kết quả.'
        shutil.copytree(previous, OUTPUT_DIR, dirs_exist_ok=True)
TRAIN_CONFIG = dict(seed=TRAIN_SEED, epochs=EPOCHS, batch_size=BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE, max_length=MAX_LENGTH, lr=LR, alpha=ALPHA,
    gamma=GAMMA, num_negsample=NUM_NEGSAMPLE, warm_up=WARM_UP,
    pretrained_model=PRETRAINED_MODEL, description_model=DESCRIPTION_MODEL,
    relation_description=RELATION_DESCRIPTION, visible_device=VISIBLE_DEVICES)
source_paths = sorted((REPO_DIR / 'model').glob('*.py')) + [REPO_DIR / 'requirements.txt']
source_paths += sorted((REL2ID_ROOT / 'fewrel_rel2id').glob('*.json'))
source_paths += [DATA_ROOT / 'fewrel/relation_description' / RELATION_DESCRIPTION]
IDENTITY = dict(config=TRAIN_CONFIG, dataset_sha256=file_sha256(dataset_target),
                source_sha256={str(p.relative_to(REPO_DIR)) if p.is_relative_to(REPO_DIR) else p.name:
                               file_sha256(p) for p in source_paths}, environment=ENVIRONMENT)
SIGNATURE = hashlib.sha256(json.dumps(IDENTITY, sort_keys=True).encode()).hexdigest()
manifest_path = OUTPUT_DIR / 'manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text())['signature'] == SIGNATURE, (
        'Code/data/config/environment đã đổi. Dùng OUTPUT_DIR mới để giữ kết quả cũ.'
    )
else:
    manifest_path.write_text(json.dumps(dict(signature=SIGNATURE, identity=IDENTITY,
        git_commit=subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip(),
        created_utc=datetime.now(timezone.utc).isoformat()), indent=2))
with (OUTPUT_DIR / 'pip-freeze.txt').open('w') as handle:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=handle, check=True)
print('Output:', OUTPUT_DIR)
print('Run signature:', SIGNATURE)


In [ ]:
METRICS_PATTERN = re.compile(r'\[test\] final precision: ([0-9.]+), recall: ([0-9.]+), f1 score: ([0-9.]+)')

def atomic_json(path, payload):
    temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(payload, indent=2))
    temp.replace(path)

def run_one(m, split, smoke=False):
    run_dir = OUTPUT_DIR / ('smoke' if smoke else 'runs') / f'm{m}_split_{split}'
    run_dir.mkdir(parents=True, exist_ok=True)
    result_file = run_dir / 'result.json'
    if result_file.exists():
        result = json.loads(result_file.read_text())
        if (result.get('status') == 'completed' and result.get('signature') == SIGNATURE
                and (run_dir / 'ckpt' / result['checkpoint_name']).is_file()):
            print(f'SKIP completed: m={m}, split={split}, smoke={smoke}')
            return result
    checkpoint_dir = run_dir / 'ckpt'
    command = [sys.executable, '-u', str(REPO_DIR / 'model/train.py'),
        '--device', 'cuda', '--dataset', 'fewrel', '--unseen', str(m),
        '--rel_split_seed', split, '--dataset_path', str(DATA_ROOT),
        '--rel2id_path', str(REL2ID_ROOT), '--ckpt_dir', str(checkpoint_dir)]
    for name, value in TRAIN_CONFIG.items():
        command.extend(['--' + name, str(value)])
    if smoke:
        command += ['--max_steps', '2']
    status = dict(status='running', m=m, split=split, smoke=smoke, signature=SIGNATURE,
                  command=command, started_utc=datetime.now(timezone.utc).isoformat())
    atomic_json(result_file, status)
    started = time.monotonic()
    started_wall_ns = time.time_ns()
    env = dict(os.environ, TOKENIZERS_PARALLELISM='false', PYTHONUNBUFFERED='1',
               PYTHONHASHSEED=str(TRAIN_SEED))
    log_path = run_dir / 'train.log'
    print(f"START m={m}, split={split}, smoke={smoke} → {log_path}")
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
        try:
            for line in process.stdout:
                print(line, end='')
                log.write(line)
                log.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            status.update(status='interrupted', elapsed_seconds=time.monotonic() - started)
            atomic_json(result_file, status)
            raise
    status['elapsed_seconds'] = time.monotonic() - started
    log_text = log_path.read_text(encoding='utf-8')
    matches = METRICS_PATTERN.findall(log_text)
    checkpoints = list(checkpoint_dir.glob(f'fewrel_split_{split}_unseen_{m}_f1_*'))
    checkpoints = [p for p in checkpoints if p.is_file() and not p.name.endswith('.json')]
    # Require a newly written checkpoint/sidecar pair for this execution.
    checkpoints = [p for p in checkpoints if p.with_name(p.name + '.json').exists() and p.stat().st_mtime_ns >= started_wall_ns]
    if returncode != 0 or not matches or not checkpoints:
        status.update(status='failed', returncode=returncode)
        atomic_json(result_file, status)
        raise RuntimeError(f'Train thất bại: m={m}, split={split}. Xem {log_path}. Không tự đổi cấu hình.')
    checkpoint = max(checkpoints, key=lambda p: p.stat().st_mtime_ns)
    precision, recall, _ = map(float, matches[-1])
    # Checkpoint filename retains full F1 precision; log P/R have 4 decimals.
    f1 = float(checkpoint.name.split('_f1_', 1)[1])
    assert all(0 <= value <= 1 for value in [precision, recall, f1])
    status.update(status='completed', precision=precision, recall=recall, f1=f1,
                  checkpoint_name=checkpoint.name, finished_utc=datetime.now(timezone.utc).isoformat())
    atomic_json(result_file, status)
    return status

def export_results():
    results = []
    for m in [5, 15]:
        for split in ['ori', '40', '41', '42', '43']:
            path = OUTPUT_DIR / 'runs' / f'm{m}_split_{split}' / 'result.json'
            if path.exists():
                result = json.loads(path.read_text())
                if result.get('status') == 'completed' and result.get('signature') == SIGNATURE:
                    results.append(result)
    fields = ['m', 'split', 'precision', 'recall', 'f1', 'elapsed_seconds', 'checkpoint_name']
    with (OUTPUT_DIR / 'results.csv').open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(results)
    summaries = []
    for m in M_VALUES:
        group = [r for r in results if r['m'] == m]
        summary = dict(m=m, completed_runs=len(group), expected_runs=5,
                       complete=len(group) == 5)
        for metric in ['precision', 'recall', 'f1']:
            values = [100 * r[metric] for r in group]
            summary[metric + '_mean_pct'] = statistics.mean(values) if values else None
            summary[metric + '_std_pct'] = statistics.stdev(values) if len(values) > 1 else None
        summaries.append(summary)
    atomic_json(OUTPUT_DIR / 'summary.json', summaries)
    with (OUTPUT_DIR / 'summary.csv').open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(summaries[0]))
        writer.writeheader()
        writer.writerows(summaries)
    return summaries


## 4. Smoke test và chạy 5 lượt cho m = 5
Smoke test vẫn đánh giá toàn bộ test split. Nếu OOM, dừng và sửa `BATCH_SIZE` trong cấu hình,
dùng thư mục output mới rồi chạy lại. Để chia công việc ra nhiều phiên, chọn `M_VALUES = [5]`
hoặc `[15]`, hoặc rút gọn `SPLITS`. Chỉ coi kết quả là trung bình 5 lần khi đủ cả 5 split.


In [ ]:
if RUN_SMOKE_TEST:
    run_one(M_VALUES[0], SPLITS[0], smoke=True)

for m in M_VALUES:
    for split in SPLITS:
        try:
            run_one(m, split)
        finally:
            # Keep completed results available even if the next run fails/is interrupted.
            export_results()
print('Đã hoàn tất các lượt được chọn.')


## 5. Kết quả và lưu outputs
`results.csv` dùng thang 0–1; `summary.csv` dùng phần trăm và sample std (ddof=1).
Precision/recall được lấy từ log đã làm tròn 4 chữ số; F1 lấy đủ độ chính xác từ tên checkpoint.
Smoke test không được tính vào kết quả. Trung bình F1 là trung bình F1 từng lượt,
không tính lại từ precision/recall trung bình.

Muốn gộp với m=10 đã chạy: giữ riêng cấu hình và 5 split của m=10 để so sánh tương ứng;
không trộn một kết quả m=10 đơn lẻ vào trung bình của m=5 hoặc m=15.


In [ ]:
summary = export_results()
for row in summary:
    print(f"m={row['m']}: {row['completed_runs']}/{row['expected_runs']} runs")
    if row['completed_runs']:
        for metric in ['precision', 'recall', 'f1']:
            mean = row[metric + '_mean_pct']
            std = row[metric + '_std_pct']
            suffix = f' ± {std:.2f}' if std is not None else ''
            print(f'  {metric}: {mean:.2f}{suffix} %')
    if not row['complete']:
        print('  Chưa đủ 5 split; không coi đây là kết quả đầy đủ.')
print('Files:', OUTPUT_DIR / 'results.csv', OUTPUT_DIR / 'summary.csv', OUTPUT_DIR / 'summary.json')
print('Lưu Kaggle Version kèm outputs để giữ log/checkpoint và tiếp tục ở phiên sau.')
